## 2.3 Analytics & Intelligence (Rule-based MCC mapping)

This notebook generates high-level insights for a **single user** (set `CLIENT_ID` below) from the cleaned/enriched artifacts:
- Spending patterns and habits
- Category-wise expenditure (rule-based via MCC)
- Budget utilization tracking

Requirement: **rule-based approach** (no LLM) to map spending to categories using the transaction MCC (`mcc_code` / `mcc_description`).

Inputs (read-only):
- `artifacts/transactions_enriched.json` (NDJSON for all users)
- Optional (if you produced it): `artifacts/transactions_enriched_{CLIENT_ID}.json`

Outputs (flat files under `artifacts/`):
- `artifacts/spend_by_category_{CLIENT_ID}.json`
- `artifacts/spend_by_mcc_{CLIENT_ID}.json`
- `artifacts/budget_utilization_{CLIENT_ID}.json`
- `artifacts/spending_patterns_{CLIENT_ID}.json`


## Imports

Uses pandas for analysis and JSON output.

In [9]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path
from typing import Any, Iterable

import pandas as pd


## Config

Set the user to analyze. If `AS_OF_DATE` is not set, this uses the latest transaction date for that user.

In [10]:
CLIENT_ID = 1696
AS_OF_DATE: str | None = None  # e.g. "2018-02-28" (optional override)


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir() and (candidate / "artifacts").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing 'data/' and 'artifacts/'")


ROOT = find_project_root()
ARTIFACTS_DIR = ROOT / "artifacts"

FOCUS_JSON = ARTIFACTS_DIR / f"transactions_enriched_{CLIENT_ID}.json"
ALL_NDJSON = ARTIFACTS_DIR / "transactions_enriched.json"

OUT_SPEND_BY_CATEGORY = ARTIFACTS_DIR / f"spend_by_category_{CLIENT_ID}.json"
OUT_SPEND_BY_MCC = ARTIFACTS_DIR / f"spend_by_mcc_{CLIENT_ID}.json"
OUT_BUDGET_UTILIZATION = ARTIFACTS_DIR / f"budget_utilization_{CLIENT_ID}.json"
OUT_SPENDING_PATTERNS = ARTIFACTS_DIR / f"spending_patterns_{CLIENT_ID}.json"

print("ROOT:", ROOT)
print("FOCUS_JSON:", FOCUS_JSON)
print("ALL_NDJSON:", ALL_NDJSON)


ROOT: /Users/nicholasp/Personal Coding/JHU/personal finance
FOCUS_JSON: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/transactions_enriched_1696.json
ALL_NDJSON: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/transactions_enriched.json


## Rule-based MCC → Category mapping

This is intentionally deterministic and explainable:
- Prefer keyword rules on `mcc_description`.
- Fall back to coarse MCC-prefix rules when needed.

Assumption: the cleaned dataset already contains `mcc_code` and `mcc_description`.

In [11]:
CATEGORY_RULES: list[tuple[str, list[str]]] = [
    ("Dining", ["restaurant", "fast food", "drinking places", "eating places"]),
    ("Groceries", ["grocery", "supermarkets", "miscellaneous food stores"]),
    ("Utilities", ["utilities", "electric", "gas", "water", "sanitary"]),
    ("Transportation", ["service stations", "tolls", "taxicabs", "limousines", "parking"]),
    ("Entertainment", ["amusement", "motion picture", "theaters", "video"]),
    ("Shopping", ["department stores", "discount stores", "book stores", "home furnishing", "lumber"]),
    ("Transfers", ["money transfer", "transfer"]),
    ("Subscriptions", ["subscription"]),
    ("Fees & Interest", ["fee", "interest"]),
]


def mcc_to_category(mcc_code: Any, mcc_description: Any) -> str:
    """Deterministic category mapping using MCC description + simple fallbacks."""
    desc = (str(mcc_description).strip().lower() if mcc_description is not None else "")
    code = str(mcc_code).strip() if mcc_code is not None else ""

    if not desc and not code:
        return "Other/Uncategorized"

    for category, keywords in CATEGORY_RULES:
        if any(k in desc for k in keywords):
            return category

    # Coarse prefix-based fallback (kept conservative)
    if code.startswith("54"):
        return "Groceries"
    if code.startswith("58"):
        return "Dining"
    if code.startswith("49"):
        return "Utilities"
    if code.startswith("41") or code.startswith("47"):
        return "Transportation"
    if code.startswith("53") or code.startswith("59") or code.startswith("52") or code.startswith("57"):
        return "Shopping"

    return "Other/Uncategorized"


# Discretionary mapping (budget utilization uses discretionary spend)
DISCRETIONARY_CATEGORIES = {
    "Dining",
    "Entertainment",
    "Shopping",
    "Travel",
    "Subscriptions",
}


def is_discretionary(category: str) -> bool:
    return category in DISCRETIONARY_CATEGORIES


## Load user data

Prefers the focus JSON (`transactions_enriched_{client_id}.json`) for speed. Falls back to scanning all-users NDJSON if needed.

In [12]:
def load_focus_records() -> list[dict[str, Any]]:
    if FOCUS_JSON.exists():
        return json.loads(FOCUS_JSON.read_text(encoding="utf-8"))

    if not ALL_NDJSON.exists():
        raise FileNotFoundError(
            "No cleaned artifacts found. Run data_processing/clean.ipynb first."
        )

    # NDJSON scan (slower)
    records: list[dict[str, Any]] = []
    with ALL_NDJSON.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if str(obj.get("client_id")) == str(CLIENT_ID):
                records.append(obj)
    return records


records = load_focus_records()
if not records:
    raise RuntimeError(f"No records found for client_id={CLIENT_ID}")

df = pd.DataFrame.from_records(records)

# Ensure types
df["amount_usd"] = pd.to_numeric(df["amount_usd"], errors="coerce")
df["transaction_dt"] = pd.to_datetime(df["transaction_dt"], errors="coerce")

print("rows:", len(df))
try:
    display(df.head(5))
except NameError:
    print(df.head(5))


rows: 30672


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,card_on_dark_web,credit_limit_usd,current_age,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,monthly_discretionary_limits,monthly_discretionary_limit_usd
0,7475539,2010-01-01 05:38:00,1696,2408,$4.02,Swipe Transaction,35451,Merritt Island,FL,32952.0,...,False,12047.0,63,"$26,339","$53,702","$85,160",606,1,"$2,238",2238.0
1,7475586,2010-01-01 06:03:00,1696,2408,$9.68,Online Transaction,39021,ONLINE,None,None,...,False,12047.0,63,"$26,339","$53,702","$85,160",606,1,"$2,238",2238.0
2,7475755,2010-01-01 06:53:00,1696,2408,$3.41,Swipe Transaction,75781,Merritt Island,FL,32953.0,...,False,12047.0,63,"$26,339","$53,702","$85,160",606,1,"$2,238",2238.0
3,7477220,2010-01-01 12:18:00,1696,2408,$9.94,Online Transaction,50404,ONLINE,None,None,...,False,12047.0,63,"$26,339","$53,702","$85,160",606,1,"$2,238",2238.0
4,7477493,2010-01-01 13:11:00,1696,2408,$-89.00,Swipe Transaction,61195,Merritt Island,FL,32952.0,...,False,12047.0,63,"$26,339","$53,702","$85,160",606,1,"$2,238",2238.0


## Compute category + MCC aggregations

In [13]:
df["category"] = df.apply(lambda r: mcc_to_category(r.get("mcc_code"), r.get("mcc_description")), axis=1)
df["is_discretionary"] = df["category"].map(is_discretionary)

# Spend by category (use positive spend only for these summaries)
spend_df = df.copy()
spend_df = spend_df.loc[spend_df["amount_usd"].notna()].copy()
spend_df["spend_usd"] = spend_df["amount_usd"].where(spend_df["amount_usd"] > 0, 0.0)

by_category = (
    spend_df.groupby("category", dropna=False)["spend_usd"].sum().sort_values(ascending=False)
)

by_mcc = (
    spend_df.groupby(["mcc_code", "mcc_description"], dropna=False)["spend_usd"]
    .sum()
    .sort_values(ascending=False)
)

spend_by_category = {k: float(v) for k, v in by_category.items()}
spend_by_mcc = [
    {
        "mcc_code": (None if pd.isna(k[0]) else str(k[0])),
        "mcc_description": (None if pd.isna(k[1]) else str(k[1])),
        "spend_usd": float(v),
    }
    for k, v in by_mcc.items()
]

print("Top categories:")
print(list(spend_by_category.items())[:10])


Top categories:
[('Transportation', 701151.12), ('Groceries', 589523.57), ('Other/Uncategorized', 190320.23), ('Shopping', 72211.4), ('Utilities', 38634.82), ('Transfers', 23460.0), ('Dining', 16517.88), ('Entertainment', 2737.09)]


## Spending patterns (habits)

Basic patterns for v1: by month, day-of-week, and top merchants (by `merchant_id`).

In [14]:
spend_df = spend_df.loc[spend_df["transaction_dt"].notna()].copy()
spend_df["month"] = spend_df["transaction_dt"].dt.to_period("M").astype(str)
spend_df["dow"] = spend_df["transaction_dt"].dt.day_name()

by_month = (
    spend_df.groupby("month")["spend_usd"].sum().sort_index()
)
by_dow = spend_df.groupby("dow")["spend_usd"].sum()

top_merchants = (
    spend_df.groupby(["merchant_id", "merchant_city", "merchant_state"], dropna=False)["spend_usd"]
    .sum()
    .sort_values(ascending=False)
    .head(25)
)

patterns = {
    "client_id": int(CLIENT_ID),
    "total_spend_usd": float(spend_df["spend_usd"].sum()),
    "by_month_usd": {k: float(v) for k, v in by_month.items()},
    "by_day_of_week_usd": {k: float(v) for k, v in by_dow.items()},
    "top_merchants_usd": [
        {
            "merchant_id": (None if pd.isna(k[0]) else str(k[0])),
            "merchant_city": (None if pd.isna(k[1]) else str(k[1])),
            "merchant_state": (None if pd.isna(k[2]) else str(k[2])),
            "spend_usd": float(v),
        }
        for k, v in top_merchants.items()
    ],
}


## Budget utilization tracking

Uses `monthly_discretionary_limit_usd` from the enriched dataset and counts spend in discretionary categories only.

In [15]:
# Determine AS_OF_DATE
max_dt = spend_df["transaction_dt"].max()
if max_dt is pd.NaT:
    raise RuntimeError("No valid transaction_dt values")

as_of = pd.to_datetime(AS_OF_DATE) if AS_OF_DATE else max_dt.normalize()
month_start = as_of.replace(day=1)

monthly_limit = pd.to_numeric(df.get("monthly_discretionary_limit_usd"), errors="coerce").dropna().iloc[0]

mtd = spend_df.loc[(spend_df["transaction_dt"] >= month_start) & (spend_df["transaction_dt"] <= as_of)].copy()
mtd_discretionary = mtd.loc[mtd["is_discretionary"]]

mtd_discretionary_spend = float(mtd_discretionary["spend_usd"].sum())
utilization_pct = float(mtd_discretionary_spend / monthly_limit) if monthly_limit else None

budget = {
    "client_id": int(CLIENT_ID),
    "as_of_date": as_of.strftime("%Y-%m-%d"),
    "month_start": month_start.strftime("%Y-%m-%d"),
    "monthly_discretionary_limit_usd": float(monthly_limit),
    "mtd_discretionary_spend_usd": mtd_discretionary_spend,
    "utilization_pct": utilization_pct,
    "discretionary_categories": sorted(list(DISCRETIONARY_CATEGORIES)),
}

budget


{'client_id': 1696,
 'as_of_date': '2018-02-28',
 'month_start': '2018-02-01',
 'monthly_discretionary_limit_usd': 2238.0,
 'mtd_discretionary_spend_usd': 626.5199999999999,
 'utilization_pct': 0.27994638069705086,
 'discretionary_categories': ['Dining',
  'Entertainment',
  'Shopping',
  'Subscriptions',
  'Travel']}

## Write artifacts (flat files under `artifacts/`)

In [16]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

OUT_SPEND_BY_CATEGORY.write_text(
    json.dumps({"client_id": int(CLIENT_ID), "spend_by_category_usd": spend_by_category}, indent=2),
    encoding="utf-8",
)
OUT_SPEND_BY_MCC.write_text(
    json.dumps({"client_id": int(CLIENT_ID), "spend_by_mcc_usd": spend_by_mcc}, indent=2),
    encoding="utf-8",
)
OUT_BUDGET_UTILIZATION.write_text(json.dumps(budget, indent=2), encoding="utf-8")
OUT_SPENDING_PATTERNS.write_text(json.dumps(patterns, indent=2), encoding="utf-8")

print("Wrote:", OUT_SPEND_BY_CATEGORY)
print("Wrote:", OUT_SPEND_BY_MCC)
print("Wrote:", OUT_BUDGET_UTILIZATION)
print("Wrote:", OUT_SPENDING_PATTERNS)


Wrote: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/spend_by_category_1696.json
Wrote: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/spend_by_mcc_1696.json
Wrote: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/budget_utilization_1696.json
Wrote: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/spending_patterns_1696.json
